# 01 完整版 Exploratory Data Analysis

这个 notebook 面向 **Home Credit Default Risk** 的 Phase 1，目标是把 `application_train.csv` 的主表分析做完整，并在相关历史表存在时补充轻量级的多表描述分析。

标准产物会保存到 `data/processed/eda/`，图表默认导出到 `reports/figures/phase1_*`。

本 notebook 的边界：
- 先做主表的完整 EDA
- 再做历史表的条件式 EDA
- 只做描述分析，不做客户级 join
- 不进入特征工程、训练表构建或建模


In [ ]:
import json
from pathlib import Path
import sys
import warnings

root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "pyproject.toml").exists()),
    Path.cwd(),
)
src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from credit_visable.config import load_settings
from credit_visable.data import load_application_train, load_table, memory_usage_mb, summarize_table_availability
from credit_visable.governance.fairness import fairness_report_placeholder
from credit_visable.utils import apply_report_style, get_paths

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", context="notebook")
apply_report_style()
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 60)


## 1. Phase Gate 与运行参数

这一步先确认原始数据路径、可用表、读取规模和导图策略。`application_train.csv` 不存在时，主表和多表分析都应顺序跳过，但 notebook 不能报错中断。


In [ ]:
settings = load_settings()
paths = get_paths()

RAW_DATA_DIR = paths.data_raw
APPLICATION_NROWS = None
HISTORY_TABLE_NROWS = 200_000
TOP_CATEGORY_N = 10
EXPORT_FIGURES = True
FIGURE_DIR = paths.reports_figures
FIGURE_PREFIX = "phase1_"
EDA_OUTPUT_ROOT = paths.data_processed / "eda"

table_summary = summarize_table_availability(data_dir=RAW_DATA_DIR, settings=settings)
available_table_names = set(table_summary.loc[table_summary["available"], "table_name"])
PHASE_1_READY = "application_train" in available_table_names

application_train = None
analysis_df = None
main_table_summary = pd.DataFrame()
duplicate_key_summary = pd.DataFrame()
derived_note_frame = pd.DataFrame()
dtype_summary = pd.DataFrame()
availability_df = pd.DataFrame()
target_summary = pd.DataFrame()
top_missing = pd.DataFrame()
numeric_summary = pd.DataFrame()
grouped_numeric_summary = pd.DataFrame()
corr_df = pd.DataFrame()
outlier_summary = pd.DataFrame()
fairness_summary = pd.DataFrame()
age_summary = pd.DataFrame()
ratio_summary = pd.DataFrame()
history_table_overview = pd.DataFrame()
checkpoint = pd.DataFrame()
phase1_artifact_manifest = pd.DataFrame()
bureau = None
bureau_balance = None
previous_application = None
installments_payments = None
credit_card_balance = None
pos_cash_balance = None

parameter_summary = pd.DataFrame(
    [
        {"parameter": "RAW_DATA_DIR", "value": str(RAW_DATA_DIR)},
        {"parameter": "APPLICATION_NROWS", "value": "full dataset" if APPLICATION_NROWS is None else APPLICATION_NROWS},
        {"parameter": "HISTORY_TABLE_NROWS", "value": HISTORY_TABLE_NROWS},
        {"parameter": "TOP_CATEGORY_N", "value": TOP_CATEGORY_N},
        {"parameter": "EXPORT_FIGURES", "value": EXPORT_FIGURES},
        {"parameter": "FIGURE_DIR", "value": str(FIGURE_DIR)},
        {"parameter": "FIGURE_PREFIX", "value": FIGURE_PREFIX},
        {"parameter": "EDA_OUTPUT_ROOT", "value": str(EDA_OUTPUT_ROOT)},
        {"parameter": "target_column", "value": settings.target_column},
        {"parameter": "id_column", "value": settings.id_column},
    ]
)

display(table_summary)
display(parameter_summary)

if PHASE_1_READY:
    print("Phase 1 ready: application_train.csv 已存在，可以执行主表 EDA。")
else:
    print("Phase 1 stop: application_train.csv 缺失。")
    print("请先完成 notebooks/00_colab_setup.ipynb，或把所需 CSV 放到 data/raw/ 后再运行本 notebook。")


## 2. Notebook 内部 Helper

为了让主表和历史表的分析结构一致，这里只在 notebook 内部定义复用函数，不修改 `src/` 模块。


In [ ]:
def maybe_save_figure(filename: str) -> None:
    if not EXPORT_FIGURES:
        return
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    resolved_filename = filename if filename.startswith(FIGURE_PREFIX) else f"{FIGURE_PREFIX}{filename}"
    output_path = FIGURE_DIR / resolved_filename
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    print(f"Figure exported to: {output_path}")


def existing_columns(frame: pd.DataFrame, columns: list[str]) -> list[str]:
    return [column for column in columns if column in frame.columns]


def safe_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    denominator = denominator.replace({0: np.nan})
    return numerator / denominator


def summarize_frame(frame: pd.DataFrame, table_name: str) -> pd.DataFrame:
    missing_share = float(frame.isna().mean().mean()) if not frame.empty else 0.0
    return pd.DataFrame(
        [
            {"metric": "table_name", "value": table_name},
            {"metric": "rows", "value": int(frame.shape[0])},
            {"metric": "columns", "value": int(frame.shape[1])},
            {"metric": "memory_mb", "value": round(memory_usage_mb(frame), 2)},
            {"metric": "average_missing_share", "value": round(missing_share, 4)},
        ]
    )


def duplicate_summary(frame: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    rows = []
    for key in keys:
        if key in frame.columns:
            rows.append(
                {
                    "key": key,
                    "present": True,
                    "duplicated_rows": int(frame[key].duplicated().sum()),
                    "unique_values": int(frame[key].nunique(dropna=True)),
                    "missing_values": int(frame[key].isna().sum()),
                }
            )
        else:
            rows.append(
                {
                    "key": key,
                    "present": False,
                    "duplicated_rows": None,
                    "unique_values": None,
                    "missing_values": None,
                }
            )
    return pd.DataFrame(rows)


def load_optional_table(table_name: str, nrows: int | None = HISTORY_TABLE_NROWS) -> pd.DataFrame | None:
    if table_name not in available_table_names:
        print(f"{table_name}: not found in the current raw-data directory. Skipping.")
        return None
    frame = load_table(table_name, data_dir=RAW_DATA_DIR, settings=settings, nrows=nrows)
    print(f"{table_name}: loaded {frame.shape[0]:,} rows x {frame.shape[1]:,} 列。")
    return frame


def plot_missingness(frame: pd.DataFrame, title: str, top_n: int = 20, filename: str | None = None) -> pd.DataFrame:
    missing_df = (
        frame.isna().mean()
        .sort_values(ascending=False)
        .head(top_n)
        .rename("missing_share")
        .reset_index()
        .rename(columns={"index": "column"})
    )
    missing_df["missing_pct"] = (missing_df["missing_share"] * 100).round(2)

    fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(missing_df))))
    sns.barplot(data=missing_df, x="missing_pct", y="column", color="#E45756", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Missing %")
    ax.set_ylabel("Column")
    plt.tight_layout()
    maybe_save_figure(filename or "missingness.png")
    plt.show()
    return missing_df


def plot_numeric_distributions(
    frame: pd.DataFrame,
    columns: list[str],
    title: str,
    bins: int = 40,
    ncols: int = 3,
    clip_quantiles: tuple[float, float] | None = None,
    filename: str | None = None,
) -> pd.DataFrame:
    valid_columns = existing_columns(frame, columns)
    if not valid_columns:
        print(f"{title}: 没有可用数值字段。")
        return pd.DataFrame()

    nrows = int(np.ceil(len(valid_columns) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).flatten()
    summary_rows = []

    for ax, column in zip(axes, valid_columns):
        series = pd.to_numeric(frame[column], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if series.empty:
            ax.text(0.5, 0.5, f"{column}\nNo plottable data", ha="center", va="center")
            ax.set_axis_off()
            continue

        plot_series = series.copy()
        if clip_quantiles is not None:
            lower = plot_series.quantile(clip_quantiles[0])
            upper = plot_series.quantile(clip_quantiles[1])
            plot_series = plot_series.clip(lower=lower, upper=upper)

        sns.histplot(plot_series, kde=True, bins=bins, color="#4C78A8", ax=ax)
        ax.set_title(column)
        ax.set_xlabel(column)
        ax.set_ylabel("Count")

        summary_rows.append(
            {
                "column": column,
                "non_null_count": int(series.shape[0]),
                "missing_share": round(float(frame[column].isna().mean()), 4) if column in frame.columns else None,
                "p01": round(float(series.quantile(0.01)), 4),
                "median": round(float(series.median()), 4),
                "p99": round(float(series.quantile(0.99)), 4),
            }
        )

    for ax in axes[len(valid_columns):]:
        ax.set_axis_off()

    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    maybe_save_figure(filename or "numeric_distributions.png")
    plt.show()
    return pd.DataFrame(summary_rows)


def plot_boxplots_by_target(
    frame: pd.DataFrame,
    columns: list[str],
    target_column: str,
    title: str,
    ncols: int = 3,
    filename: str | None = None,
) -> None:
    if target_column not in frame.columns:
        print(f"{title}: target 列不存在，无法按 TARGET 对比。")
        return

    valid_columns = existing_columns(frame, columns)
    if not valid_columns:
        print(f"{title}: 没有可用数值字段。")
        return

    nrows = int(np.ceil(len(valid_columns) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for ax, column in zip(axes, valid_columns):
        plot_df = frame[[target_column, column]].copy()
        plot_df[column] = pd.to_numeric(plot_df[column], errors="coerce")
        plot_df = plot_df.replace([np.inf, -np.inf], np.nan).dropna()
        if plot_df.empty:
            ax.text(0.5, 0.5, f"{column}\nNo plottable data", ha="center", va="center")
            ax.set_axis_off()
            continue
        sns.boxplot(data=plot_df, x=target_column, y=column, color="#72B7B2", ax=ax)
        ax.set_title(column)
        ax.set_xlabel(target_column)
        ax.set_ylabel(column)

    for ax in axes[len(valid_columns):]:
        ax.set_axis_off()

    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    maybe_save_figure(filename or "boxplots_by_target.png")
    plt.show()


def plot_top_categories(
    frame: pd.DataFrame,
    column: str,
    target_column: str,
    top_n: int = TOP_CATEGORY_N,
    filename: str | None = None,
) -> pd.DataFrame:
    if column not in frame.columns:
        print(f"{column}: 字段不存在，跳过。")
        return pd.DataFrame()

    plot_df = frame[[column]].copy()
    plot_df[column] = plot_df[column].fillna("(Missing)").astype(str)
    top_values = plot_df[column].value_counts().head(top_n).index.tolist()
    plot_df = plot_df.loc[plot_df[column].isin(top_values)].copy()

    count_summary = (
        plot_df[column]
        .value_counts()
        .rename_axis("group")
        .reset_index(name="count")
    )

    if target_column in frame.columns:
        plot_df[target_column] = frame.loc[plot_df.index, target_column]
        rate_summary = (
            plot_df.groupby(column, dropna=False)[target_column]
            .agg(["count", "mean"])
            .reset_index()
            .rename(columns={column: "group", "mean": "target_rate"})
            .sort_values("count", ascending=False)
        )
    else:
        rate_summary = count_summary.copy()
        rate_summary["target_rate"] = np.nan

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=count_summary, x="count", y="group", color="#4C78A8", ax=axes[0])
    axes[0].set_title(f"{column}: Top {top_n} Count")
    axes[0].set_xlabel("Count")
    axes[0].set_ylabel(column)

    if rate_summary["target_rate"].notna().any():
        rate_plot = rate_summary.copy()
        rate_plot["target_rate_pct"] = rate_plot["target_rate"] * 100
        sns.barplot(data=rate_plot, x="target_rate_pct", y="group", color="#F58518", ax=axes[1])
        axes[1].set_title(f"{column}: Group Default Rate")
        axes[1].set_xlabel("Default rate (%)")
        axes[1].set_ylabel(column)
    else:
        axes[1].text(0.5, 0.5, "TARGET is unavailable, so default-rate views cannot be computed", ha="center", va="center")
        axes[1].set_axis_off()

    plt.tight_layout()
    maybe_save_figure(filename or f"{column.lower()}_category_analysis.png")
    plt.show()
    return rate_summary


def grouped_default_rate(frame: pd.DataFrame, group_column: str, target_column: str) -> pd.DataFrame:
    if group_column not in frame.columns or target_column not in frame.columns:
        return pd.DataFrame()
    summary = (
        frame.groupby(group_column, dropna=False)[target_column]
        .agg(["count", "mean"])
        .reset_index()
        .rename(columns={group_column: "group", "mean": "target_rate"})
        .sort_values("count", ascending=False)
    )
    return summary


def plot_grouped_default_rate(summary: pd.DataFrame, title: str, filename: str | None = None) -> None:
    if summary.empty or "target_rate" not in summary.columns:
        print(f"{title}: 没有可绘制的坏账率汇总。")
        return
    plot_df = summary.copy()
    plot_df["group"] = plot_df["group"].astype(str)
    plot_df["target_rate_pct"] = plot_df["target_rate"] * 100
    fig, ax = plt.subplots(figsize=(10, max(4, min(8, 0.4 * len(plot_df)))))
    sns.barplot(data=plot_df, x="target_rate_pct", y="group", color="#54A24B", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Default rate (%)")
    ax.set_ylabel("Group")
    plt.tight_layout()
    maybe_save_figure(filename or "grouped_default_rate.png")
    plt.show()


## 3. 主表加载与派生字段

先加载 `application_train`，然后补充更适合业务解读的字段，比如年龄年数、就业年数和收入/授信相关比例。


In [ ]:
if PHASE_1_READY:
    application_train = load_application_train(data_dir=RAW_DATA_DIR, nrows=APPLICATION_NROWS)
    analysis_df = application_train.copy()

    derived_notes = []

    if "DAYS_BIRTH" in analysis_df.columns:
        analysis_df["AGE_YEARS"] = (-analysis_df["DAYS_BIRTH"] / 365.25).round(1)
        derived_notes.append("AGE_YEARS 已生成。")

    if "DAYS_EMPLOYED" in analysis_df.columns:
        sentinel_mask = analysis_df["DAYS_EMPLOYED"] == 365243
        sentinel_count = int(sentinel_mask.sum())
        analysis_df["DAYS_EMPLOYED_CLEAN"] = analysis_df["DAYS_EMPLOYED"].mask(sentinel_mask)
        analysis_df["YEARS_EMPLOYED"] = (-analysis_df["DAYS_EMPLOYED_CLEAN"] / 365.25).round(1)
        derived_notes.append(f"YEARS_EMPLOYED 已生成，并把 {sentinel_count:,} 个 365243 哨兵值视为缺失。")

    if {"AMT_INCOME_TOTAL", "AMT_CREDIT"}.issubset(analysis_df.columns):
        analysis_df["INCOME_CREDIT_RATIO"] = safe_ratio(analysis_df["AMT_INCOME_TOTAL"], analysis_df["AMT_CREDIT"])
        derived_notes.append("INCOME_CREDIT_RATIO 已生成。")

    if {"AMT_ANNUITY", "AMT_INCOME_TOTAL"}.issubset(analysis_df.columns):
        analysis_df["ANNUITY_INCOME_RATIO"] = safe_ratio(analysis_df["AMT_ANNUITY"], analysis_df["AMT_INCOME_TOTAL"])
        derived_notes.append("ANNUITY_INCOME_RATIO 已生成。")

    main_table_summary = summarize_frame(analysis_df, "application_train")
    duplicate_key_summary = duplicate_summary(analysis_df, [settings.id_column, settings.target_column])
    derived_note_frame = pd.DataFrame({"derived_note": derived_notes or ["没有生成额外派生字段。"]})

    display(main_table_summary)
    display(duplicate_key_summary)
    display(derived_note_frame)
    display(analysis_df.head())
else:
    application_train = None
    analysis_df = None
    print("主表加载已跳过，因为 application_train.csv 当前不可用。")


## 4. 主表深度 EDA

这一部分覆盖样本结构、字段类型、目标分布、缺失率、数值变量分布和 `EXT_SOURCE_*` 的基础信号。重点是先把单表的风险结构看清楚。


In [ ]:
if not PHASE_1_READY:
    print("主表深度 EDA 已跳过，因为 Phase 1 还未准备好。")
else:
    dtype_summary = (
        analysis_df.dtypes.astype(str)
        .value_counts()
        .rename_axis("dtype")
        .reset_index(name="column_count")
    )
    display(dtype_summary)

    key_business_columns = [
        settings.id_column,
        settings.target_column,
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "AMT_GOODS_PRICE",
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
        "CODE_GENDER",
        "NAME_INCOME_TYPE",
        "NAME_EDUCATION_TYPE",
        "NAME_FAMILY_STATUS",
    ]
    availability_df = pd.DataFrame(
        [
            {
                "column": column,
                "present": column in analysis_df.columns,
                "missing_share": round(float(analysis_df[column].isna().mean()), 4) if column in analysis_df.columns else None,
            }
            for column in key_business_columns
        ]
    )
    display(availability_df)

    if settings.target_column in analysis_df.columns:
        target_summary = (
            analysis_df[settings.target_column]
            .value_counts(dropna=False)
            .rename_axis("target_value")
            .reset_index(name="count")
            .sort_values("target_value")
            .reset_index(drop=True)
        )
        target_summary["share"] = (target_summary["count"] / target_summary["count"].sum()).round(4)
        display(target_summary)

        fig, ax = plt.subplots(figsize=(6, 4))
        sns.barplot(data=target_summary.assign(target_label=lambda df: df["target_value"].astype(str)), x="target_label", y="count", color="#4C78A8", ax=ax)
        ax.set_title("TARGET Distribution")
        ax.set_xlabel("TARGET")
        ax.set_ylabel("Application Count")
        plt.tight_layout()
        maybe_save_figure("main_target_distribution.png")
        plt.show()

        print(f"观察到的坏账率: {analysis_df[settings.target_column].mean():.2%}")
    else:
        print("主表中没有 TARGET，无法继续坏账率相关分析。")

    top_missing = plot_missingness(
        analysis_df,
        title="Main Table Top 20 Missingness",
        top_n=20,
        filename="main_missingness_top20.png",
    )
    if not top_missing.empty:
        display(top_missing[["column", "missing_pct"]])


业务解读：

- `TARGET` 的不平衡程度会直接影响后续模型评价方式，不能只看 accuracy。
- 高缺失字段可能反映资料采集摩擦、申请链路差异，或者只在特定客群中填写。
- `EXT_SOURCE_*`、收入、授信和年金类字段是主表里最值得优先关注的风险信息。


## 5. 数值变量分布与 TARGET 对比

这一部分重点看偿债能力、授信暴露、年龄结构和外部评分。为了更容易读图，分布图只在展示时对极端值做轻微截尾，不改原始数据。


In [ ]:
if not PHASE_1_READY:
    print("数值变量分布已跳过，因为主表不可用。")
else:
    main_numeric_columns = [
        "AGE_YEARS",
        "YEARS_EMPLOYED",
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "AMT_GOODS_PRICE",
        "INCOME_CREDIT_RATIO",
        "ANNUITY_INCOME_RATIO",
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
    ]
    numeric_summary = plot_numeric_distributions(
        analysis_df,
        columns=main_numeric_columns,
        title="Main Table Key Numeric Distributions (Winsorized to 1st-99th Percentile)",
        bins=40,
        ncols=3,
        clip_quantiles=(0.01, 0.99),
        filename="main_numeric_distributions.png",
    )
    display(numeric_summary)


In [ ]:
if not PHASE_1_READY:
    print("按 TARGET 的数值对比已跳过，因为主表不可用。")
else:
    target_compare_columns = [
        "AGE_YEARS",
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "INCOME_CREDIT_RATIO",
        "ANNUITY_INCOME_RATIO",
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
    ]
    plot_boxplots_by_target(
        analysis_df,
        columns=target_compare_columns,
        target_column=settings.target_column,
        title="Key Numeric Features by TARGET",
        ncols=3,
        filename="numeric_boxplots_by_target.png",
    )

    available_compare_columns = existing_columns(analysis_df, target_compare_columns)
    if settings.target_column in analysis_df.columns and available_compare_columns:
        grouped_numeric_summary = (
            analysis_df.groupby(settings.target_column)[available_compare_columns]
            .median(numeric_only=True)
            .T
            .reset_index()
            .rename(columns={"index": "column"})
        )
        display(grouped_numeric_summary)


业务解读：

- `AMT_CREDIT`、`AMT_ANNUITY` 和各类收入比例字段，能帮助判断客户的月供压力和暴露水平。
- `EXT_SOURCE_*` 通常是主表里最强的外部风险线索，值得单独看分布与分组差异。
- 年龄和就业年限不一定代表保护性或不利性结论，但它们可能携带生命周期和稳定性信号，也可能形成代理变量，需要谨慎解释。


## 6. 相关性与异常值检查

这里做一个紧凑的数值相关性热力图，并补一张分位数摘要表，帮助识别极端偏态和离群值风险。


In [ ]:
if not PHASE_1_READY:
    print("相关性与异常值检查已跳过，因为主表不可用。")
else:
    corr_candidates = [
        settings.target_column,
        "AGE_YEARS",
        "YEARS_EMPLOYED",
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "INCOME_CREDIT_RATIO",
        "ANNUITY_INCOME_RATIO",
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
    ]
    corr_columns = existing_columns(analysis_df, corr_candidates)
    if len(corr_columns) >= 2:
        corr_df = analysis_df[corr_columns].apply(pd.to_numeric, errors="coerce").corr()
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, ax=ax)
        ax.set_title("Main Table Key Numeric Feature Correlation Heatmap")
        plt.tight_layout()
        maybe_save_figure("main_correlation_heatmap.png")
        plt.show()
        display(corr_df.round(3))
    else:
        print("可用于相关性分析的字段不足两个。")

    outlier_columns = existing_columns(
        analysis_df,
        [
            "AMT_INCOME_TOTAL",
            "AMT_CREDIT",
            "AMT_ANNUITY",
            "AMT_GOODS_PRICE",
            "INCOME_CREDIT_RATIO",
            "ANNUITY_INCOME_RATIO",
            "YEARS_EMPLOYED",
        ],
    )
    outlier_rows = []
    for column in outlier_columns:
        series = pd.to_numeric(analysis_df[column], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if series.empty:
            continue
        outlier_rows.append(
            {
                "column": column,
                "missing_share": round(float(analysis_df[column].isna().mean()), 4),
                "zero_share": round(float((series == 0).mean()), 4),
                "p01": round(float(series.quantile(0.01)), 4),
                "median": round(float(series.median()), 4),
                "p99": round(float(series.quantile(0.99)), 4),
                "max": round(float(series.max()), 4),
            }
        )
    outlier_summary = pd.DataFrame(outlier_rows)
    display(outlier_summary)


## 7. 类别变量业务分析

这一部分看主表的主要类别型字段：一方面看频次结构，另一方面看坏账率分层。它能帮助判断客群差异、潜在代理特征和后续编码重点。


In [ ]:
if not PHASE_1_READY:
    print("类别变量业务分析已跳过，因为主表不可用。")
else:
    category_columns = [
        "CODE_GENDER",
        "NAME_FAMILY_STATUS",
        "NAME_EDUCATION_TYPE",
        "NAME_INCOME_TYPE",
        "OCCUPATION_TYPE",
        "ORGANIZATION_TYPE",
        "FLAG_OWN_CAR",
        "FLAG_OWN_REALTY",
    ]
    category_availability = pd.DataFrame(
        [
            {"column": column, "present": column in analysis_df.columns}
            for column in category_columns
        ]
    )
    display(category_availability)

    for column in category_columns:
        summary = plot_top_categories(
            analysis_df,
            column=column,
            target_column=settings.target_column,
            top_n=TOP_CATEGORY_N,
            filename=f"{column.lower()}_business_view.png",
        )
        if not summary.empty:
            display(summary)


业务解读：

- 类别频次让我们知道哪些客群量级足够大，适合做稳定分析；哪些类别过稀，后续可能需要合并。
- 分类坏账率差异可以帮助识别业务上值得追踪的风险分层，但不能直接等同于因果结论。
- 类似性别、婚姻、职业、组织类型等字段可能带来业务含义，也可能形成代理偏差，因此要和公平性视角一起看。


## 8. 描述性风险 / 公平性切片

这里不是正式公平性审计，只做 grouped default-rate 的描述性切片，帮助识别哪些群组在当前样本里表现出明显的风险差异。


In [ ]:
if not PHASE_1_READY:
    print("描述性风险切片已跳过，因为主表不可用。")
else:
    protected_columns = existing_columns(
        analysis_df,
        ["CODE_GENDER", "NAME_FAMILY_STATUS", "NAME_INCOME_TYPE"],
    )
    if settings.target_column in analysis_df.columns and protected_columns:
        fairness_summary = fairness_report_placeholder(
            analysis_df[protected_columns + [settings.target_column]].copy(),
            target_column=settings.target_column,
            protected_columns=protected_columns,
        )
        display(fairness_summary)
        for column in protected_columns:
            column_summary = fairness_summary.loc[
                fairness_summary["protected_attribute"] == column,
                ["group", "count", "target_rate"],
            ].copy()
            plot_grouped_default_rate(
                column_summary,
                title=f"{column} Group Default Rate (Descriptive Slice)",
                filename=f"{column.lower()}_default_rate_slice.png",
            )

    if {"AGE_YEARS", settings.target_column}.issubset(analysis_df.columns):
        age_slice = analysis_df[["AGE_YEARS", settings.target_column]].dropna().copy()
        age_slice = age_slice.loc[age_slice["AGE_YEARS"].between(18, 100)]
        if not age_slice.empty:
            age_slice["AGE_BAND"] = pd.cut(
                age_slice["AGE_YEARS"],
                bins=[18, 25, 35, 45, 55, 65, 100],
                include_lowest=True,
                right=False,
            )
            age_summary = grouped_default_rate(age_slice, "AGE_BAND", settings.target_column)
            display(age_summary)
            plot_grouped_default_rate(
                age_summary,
                title="Age Band Default Rate",
                filename="age_band_default_rate.png",
            )

    if {"ANNUITY_INCOME_RATIO", settings.target_column}.issubset(analysis_df.columns):
        ratio_slice = analysis_df[["ANNUITY_INCOME_RATIO", settings.target_column]].replace([np.inf, -np.inf], np.nan).dropna().copy()
        if not ratio_slice.empty and ratio_slice["ANNUITY_INCOME_RATIO"].nunique() >= 4:
            q = int(min(5, ratio_slice["ANNUITY_INCOME_RATIO"].nunique()))
            ratio_slice["ANNUITY_INCOME_BAND"] = pd.qcut(
                ratio_slice["ANNUITY_INCOME_RATIO"],
                q=q,
                duplicates="drop",
            )
            ratio_summary = grouped_default_rate(ratio_slice, "ANNUITY_INCOME_BAND", settings.target_column)
            display(ratio_summary)
            plot_grouped_default_rate(
                ratio_summary,
                title="Annuity-to-Income Ratio Band Default Rate",
                filename="annuity_income_ratio_default_rate.png",
            )


## 9. 多表条件式 EDA 总览

从这里开始，只在历史表真实存在时才继续。每张表都做轻量级理解，不进行客户级聚合回主表。


In [ ]:
history_table_focus = pd.DataFrame(
    [
        {"table_name": "bureau", "available": "bureau" in available_table_names, "focus": "外部贷款状态、债务与客户记录数"},
        {"table_name": "bureau_balance", "available": "bureau_balance" in available_table_names, "focus": "月度状态分布与每笔贷款月序列长度"},
        {"table_name": "previous_application", "available": "previous_application" in available_table_names, "focus": "历史申请状态、申请金额与批准金额"},
        {"table_name": "installments_payments", "available": "installments_payments" in available_table_names, "focus": "逾期天数、少还标记、客户还款记录密度"},
        {"table_name": "credit_card_balance", "available": "credit_card_balance" in available_table_names, "focus": "额度利用率、逾期天数、客户月记录数"},
        {"table_name": "pos_cash_balance", "available": "pos_cash_balance" in available_table_names, "focus": "合同状态、逾期、剩余期数"},
    ]
)
display(history_table_focus)


### 9.1 `bureau.csv`

`bureau` 帮助理解客户在其他金融机构的外部信贷画像。重点关注状态、债务和每客户记录数。


In [ ]:
bureau = load_optional_table("bureau", nrows=HISTORY_TABLE_NROWS)
if bureau is not None:
    display(summarize_frame(bureau, "bureau"))
    display(duplicate_summary(bureau, ["SK_ID_BUREAU", "SK_ID_CURR"]))
    display(bureau.head())

    if "CREDIT_ACTIVE" in bureau.columns:
        credit_active_summary = (
            bureau["CREDIT_ACTIVE"].fillna("(Missing)").astype(str)
            .value_counts()
            .rename_axis("credit_active")
            .reset_index(name="count")
        )
        display(credit_active_summary)
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.barplot(data=credit_active_summary, x="count", y="credit_active", color="#4C78A8", ax=ax)
        ax.set_title("bureau: CREDIT_ACTIVE Distribution")
        ax.set_xlabel("Count")
        ax.set_ylabel("CREDIT_ACTIVE")
        plt.tight_layout()
        maybe_save_figure("bureau_credit_active_distribution.png")
        plt.show()

    if "SK_ID_CURR" in bureau.columns:
        bureau_records_per_customer = bureau.groupby("SK_ID_CURR").size().rename("bureau_record_count")
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(bureau_records_per_customer.clip(upper=bureau_records_per_customer.quantile(0.99)), bins=40, color="#72B7B2", ax=ax)
        ax.set_title("bureau: External Loan Records per Customer (Clipped at 99th Percentile)")
        ax.set_xlabel("bureau_record_count")
        ax.set_ylabel("Customer count")
        plt.tight_layout()
        maybe_save_figure("bureau_records_per_customer.png")
        plt.show()
        display(bureau_records_per_customer.describe().to_frame(name="bureau_record_count"))

    bureau_numeric = plot_numeric_distributions(
        bureau,
        columns=["AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE", "CREDIT_DAY_OVERDUE", "CNT_CREDIT_PROLONG"],
        title="bureau: Amount and Delinquency Distributions (Winsorized to 1st-99th Percentile)",
        bins=40,
        ncols=3,
        clip_quantiles=(0.01, 0.99),
        filename="bureau_numeric_distribution.png",
    )
    display(bureau_numeric)


### 9.2 `bureau_balance.csv`

`bureau_balance` 是月度状态轨迹表。这里不做聚合，只看状态分布、月份跨度和每笔贷款的月记录长度。


In [ ]:
bureau_balance = load_optional_table("bureau_balance", nrows=HISTORY_TABLE_NROWS)
if bureau_balance is not None:
    display(summarize_frame(bureau_balance, "bureau_balance"))
    display(duplicate_summary(bureau_balance, ["SK_ID_BUREAU"]))
    display(bureau_balance.head())

    if "STATUS" in bureau_balance.columns:
        status_summary = (
            bureau_balance["STATUS"].fillna("(Missing)").astype(str)
            .value_counts()
            .rename_axis("status")
            .reset_index(name="count")
        )
        display(status_summary)
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.barplot(data=status_summary, x="count", y="status", color="#F58518", ax=ax)
        ax.set_title("bureau_balance: STATUS Distribution")
        ax.set_xlabel("Count")
        ax.set_ylabel("STATUS")
        plt.tight_layout()
        maybe_save_figure("bureau_balance_status_distribution.png")
        plt.show()

    bureau_balance_numeric = plot_numeric_distributions(
        bureau_balance,
        columns=["MONTHS_BALANCE"],
        title="bureau_balance: MONTHS_BALANCE Distribution",
        bins=40,
        ncols=1,
        filename="bureau_balance_months_balance.png",
    )
    display(bureau_balance_numeric)

    if "SK_ID_BUREAU" in bureau_balance.columns:
        monthly_record_count = bureau_balance.groupby("SK_ID_BUREAU").size().rename("monthly_record_count")
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(monthly_record_count.clip(upper=monthly_record_count.quantile(0.99)), bins=40, color="#54A24B", ax=ax)
        ax.set_title("bureau_balance: Monthly Records per Bureau Loan (Clipped at 99th Percentile)")
        ax.set_xlabel("monthly_record_count")
        ax.set_ylabel("Loan count")
        plt.tight_layout()
        maybe_save_figure("bureau_balance_monthly_record_count.png")
        plt.show()
        display(monthly_record_count.describe().to_frame(name="monthly_record_count"))


### 9.3 `previous_application.csv`

`previous_application` 反映客户过往申请行为。这里主要看状态、申请金额与批准金额的关系，以及每客户历史申请次数。


In [ ]:
previous_application = load_optional_table("previous_application", nrows=HISTORY_TABLE_NROWS)
if previous_application is not None:
    display(summarize_frame(previous_application, "previous_application"))
    display(duplicate_summary(previous_application, ["SK_ID_PREV", "SK_ID_CURR"]))
    display(previous_application.head())

    for categorical_column in ["NAME_CONTRACT_STATUS", "NAME_CONTRACT_TYPE"]:
        if categorical_column in previous_application.columns:
            summary = (
                previous_application[categorical_column].fillna("(Missing)").astype(str)
                .value_counts()
                .rename_axis(categorical_column)
                .reset_index(name="count")
            )
            display(summary)
            fig, ax = plt.subplots(figsize=(8, 4))
            sns.barplot(data=summary.head(TOP_CATEGORY_N), x="count", y=categorical_column, color="#4C78A8", ax=ax)
            ax.set_title(f"previous_application: {categorical_column} Distribution")
            ax.set_xlabel("Count")
            ax.set_ylabel(categorical_column)
            plt.tight_layout()
            maybe_save_figure(f"previous_application_{categorical_column.lower()}_distribution.png")
            plt.show()

    previous_numeric = plot_numeric_distributions(
        previous_application,
        columns=["AMT_APPLICATION", "AMT_CREDIT", "AMT_ANNUITY", "CNT_PAYMENT", "DAYS_DECISION"],
        title="previous_application: Amount, Tenor, and Decision-Time Distributions (Winsorized to 1st-99th Percentile)",
        bins=40,
        ncols=3,
        clip_quantiles=(0.01, 0.99),
        filename="previous_application_numeric_distribution.png",
    )
    display(previous_numeric)

    if {"AMT_APPLICATION", "AMT_CREDIT"}.issubset(previous_application.columns):
        scatter_df = previous_application[["AMT_APPLICATION", "AMT_CREDIT"]].dropna().copy()
        scatter_df = scatter_df.sample(min(len(scatter_df), 5000), random_state=settings.random_state)
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.scatter(scatter_df["AMT_APPLICATION"], scatter_df["AMT_CREDIT"], alpha=0.2, color="#E45756")
        ax.set_title("previous_application: Applied Amount vs Approved Credit")
        ax.set_xlabel("AMT_APPLICATION")
        ax.set_ylabel("AMT_CREDIT")
        plt.tight_layout()
        maybe_save_figure("previous_application_application_vs_credit.png")
        plt.show()

    if "SK_ID_CURR" in previous_application.columns:
        previous_count = previous_application.groupby("SK_ID_CURR").size().rename("previous_application_count")
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(previous_count.clip(upper=previous_count.quantile(0.99)), bins=40, color="#72B7B2", ax=ax)
        ax.set_title("previous_application: Historical Applications per Customer (Clipped at 99th Percentile)")
        ax.set_xlabel("previous_application_count")
        ax.set_ylabel("Customer count")
        plt.tight_layout()
        maybe_save_figure("previous_application_count_per_customer.png")
        plt.show()
        display(previous_count.describe().to_frame(name="previous_application_count"))


### 9.4 `installments_payments.csv`

`installments_payments` 更接近行为数据。这里先生成晚还天数和少还标记，再看分布和每客户逾期记录密度。


In [ ]:
installments_payments = load_optional_table("installments_payments", nrows=HISTORY_TABLE_NROWS)
if installments_payments is not None:
    installments_analysis = installments_payments.copy()
    if {"DAYS_ENTRY_PAYMENT", "DAYS_INSTALMENT"}.issubset(installments_analysis.columns):
        installments_analysis["late_days"] = installments_analysis["DAYS_ENTRY_PAYMENT"] - installments_analysis["DAYS_INSTALMENT"]
        installments_analysis["late_flag"] = (installments_analysis["late_days"] > 0).astype(int)
    if {"AMT_PAYMENT", "AMT_INSTALMENT"}.issubset(installments_analysis.columns):
        installments_analysis["pay_diff"] = installments_analysis["AMT_PAYMENT"] - installments_analysis["AMT_INSTALMENT"]
        installments_analysis["underpay_flag"] = (installments_analysis["AMT_PAYMENT"] < installments_analysis["AMT_INSTALMENT"]).astype(int)

    display(summarize_frame(installments_analysis, "installments_payments"))
    display(duplicate_summary(installments_analysis, ["SK_ID_PREV", "SK_ID_CURR"]))
    display(installments_analysis.head())

    installments_numeric = plot_numeric_distributions(
        installments_analysis,
        columns=["late_days", "pay_diff", "AMT_PAYMENT", "AMT_INSTALMENT"],
        title="installments_payments: Repayment Behavior Distributions (Winsorized to 1st-99th Percentile)",
        bins=40,
        ncols=2,
        clip_quantiles=(0.01, 0.99),
        filename="installments_numeric_distribution.png",
    )
    display(installments_numeric)

    if {"late_flag", "underpay_flag"}.issubset(installments_analysis.columns):
        behavior_summary = pd.DataFrame(
            [
                {"metric": "late_flag_mean", "value": round(float(installments_analysis["late_flag"].mean()), 4)},
                {"metric": "underpay_flag_mean", "value": round(float(installments_analysis["underpay_flag"].mean()), 4)},
            ]
        )
        display(behavior_summary)

    if {"SK_ID_CURR", "late_flag"}.issubset(installments_analysis.columns):
        late_count_per_customer = installments_analysis.groupby("SK_ID_CURR")["late_flag"].sum().rename("late_count")
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(late_count_per_customer.clip(upper=late_count_per_customer.quantile(0.99)), bins=40, color="#F58518", ax=ax)
        ax.set_title("installments_payments: Late-Payment Count per Customer (Clipped at 99th Percentile)")
        ax.set_xlabel("late_count")
        ax.set_ylabel("Customer count")
        plt.tight_layout()
        maybe_save_figure("installments_late_count_per_customer.png")
        plt.show()
        display(late_count_per_customer.describe().to_frame(name="late_count"))


### 9.5 `credit_card_balance.csv`

`credit_card_balance` 主要看额度利用率和逾期天数。这里只做轻量派生，不把月度信息聚成客户级训练特征。


In [ ]:
credit_card_balance = load_optional_table("credit_card_balance", nrows=HISTORY_TABLE_NROWS)
if credit_card_balance is not None:
    credit_card_analysis = credit_card_balance.copy()
    if {"AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL"}.issubset(credit_card_analysis.columns):
        credit_card_analysis["utilization"] = safe_ratio(
            credit_card_analysis["AMT_BALANCE"],
            credit_card_analysis["AMT_CREDIT_LIMIT_ACTUAL"],
        )
        credit_card_analysis["high_util_flag"] = (credit_card_analysis["utilization"] > 0.8).astype(int)
    if "SK_DPD" in credit_card_analysis.columns:
        credit_card_analysis["dpd_flag"] = (credit_card_analysis["SK_DPD"] > 0).astype(int)

    display(summarize_frame(credit_card_analysis, "credit_card_balance"))
    display(duplicate_summary(credit_card_analysis, ["SK_ID_PREV", "SK_ID_CURR"]))
    display(credit_card_analysis.head())

    credit_card_numeric = plot_numeric_distributions(
        credit_card_analysis,
        columns=["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "utilization", "SK_DPD", "SK_DPD_DEF", "AMT_PAYMENT_TOTAL_CURRENT"],
        title="credit_card_balance: Balance, Utilization, and Delinquency Distributions (Winsorized to 1st-99th Percentile)",
        bins=40,
        ncols=3,
        clip_quantiles=(0.01, 0.99),
        filename="credit_card_numeric_distribution.png",
    )
    display(credit_card_numeric)

    summary_rows = []
    for column in ["high_util_flag", "dpd_flag"]:
        if column in credit_card_analysis.columns:
            summary_rows.append({"metric": column, "value": round(float(credit_card_analysis[column].mean()), 4)})
    if summary_rows:
        display(pd.DataFrame(summary_rows))

    if "SK_ID_CURR" in credit_card_analysis.columns:
        credit_card_records = credit_card_analysis.groupby("SK_ID_CURR").size().rename("credit_card_record_count")
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(credit_card_records.clip(upper=credit_card_records.quantile(0.99)), bins=40, color="#54A24B", ax=ax)
        ax.set_title("credit_card_balance: Monthly Records per Customer (Clipped at 99th Percentile)")
        ax.set_xlabel("credit_card_record_count")
        ax.set_ylabel("Customer count")
        plt.tight_layout()
        maybe_save_figure("credit_card_record_count_per_customer.png")
        plt.show()
        display(credit_card_records.describe().to_frame(name="credit_card_record_count"))


### 9.6 `POS_CASH_balance.csv`

`POS_CASH_balance` 主要反映现金贷 / 分期消费贷的月度状态，这里看合同状态、逾期和剩余期数。


In [ ]:
pos_cash_balance = load_optional_table("pos_cash_balance", nrows=HISTORY_TABLE_NROWS)
if pos_cash_balance is not None:
    display(summarize_frame(pos_cash_balance, "pos_cash_balance"))
    display(duplicate_summary(pos_cash_balance, ["SK_ID_PREV", "SK_ID_CURR"]))
    display(pos_cash_balance.head())

    if "NAME_CONTRACT_STATUS" in pos_cash_balance.columns:
        contract_status_summary = (
            pos_cash_balance["NAME_CONTRACT_STATUS"].fillna("(Missing)").astype(str)
            .value_counts()
            .rename_axis("contract_status")
            .reset_index(name="count")
        )
        display(contract_status_summary)
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.barplot(data=contract_status_summary.head(TOP_CATEGORY_N), x="count", y="contract_status", color="#4C78A8", ax=ax)
        ax.set_title("pos_cash_balance: Contract Status Distribution")
        ax.set_xlabel("Count")
        ax.set_ylabel("NAME_CONTRACT_STATUS")
        plt.tight_layout()
        maybe_save_figure("pos_cash_contract_status_distribution.png")
        plt.show()

    pos_cash_numeric = plot_numeric_distributions(
        pos_cash_balance,
        columns=["SK_DPD", "SK_DPD_DEF", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE", "MONTHS_BALANCE"],
        title="pos_cash_balance: Delinquency, Installment, and Month Distributions (Winsorized to 1st-99th Percentile)",
        bins=40,
        ncols=3,
        clip_quantiles=(0.01, 0.99),
        filename="pos_cash_numeric_distribution.png",
    )
    display(pos_cash_numeric)


## 10. Phase 1 Checkpoint

到这里，主表和可用历史表都已经做了表级理解。下一步应该是决定哪些字段和信号进入 `02_preprocessing.ipynb`，而不是在这里直接做 join 或建模。


In [ ]:
if not PHASE_1_READY:
    print("Checkpoint: 当前仍停在 Phase 0/Phase 1 交界，因为 application_train.csv 缺失。")
else:
    history_tables = [
        "bureau",
        "bureau_balance",
        "previous_application",
        "installments_payments",
        "credit_card_balance",
        "pos_cash_balance",
    ]
    available_history = [table for table in history_tables if table in available_table_names]
    missing_history = [table for table in history_tables if table not in available_table_names]

    checkpoint = pd.DataFrame(
        [
            {"item": "application_train_rows", "value": int(analysis_df.shape[0])},
            {"item": "application_train_columns", "value": int(analysis_df.shape[1])},
            {
                "item": "observed_default_rate",
                "value": round(float(analysis_df[settings.target_column].mean()), 4)
                if settings.target_column in analysis_df.columns
                else None,
            },
            {"item": "available_history_tables", "value": ", ".join(available_history) if available_history else "none"},
            {"item": "missing_history_tables", "value": ", ".join(missing_history) if missing_history else "none"},
            {"item": "export_figures", "value": EXPORT_FIGURES},
            {"item": "next_notebook", "value": "notebooks/02_preprocessing.ipynb"},
        ]
    )
    display(checkpoint)

    history_frame_map = {
        "bureau": bureau,
        "bureau_balance": bureau_balance,
        "previous_application": previous_application,
        "installments_payments": installments_payments,
        "credit_card_balance": credit_card_balance,
        "pos_cash_balance": pos_cash_balance,
    }
    history_table_overview = pd.DataFrame(
        [
            {
                "table_name": table_name,
                "loaded": frame is not None,
                "rows": int(frame.shape[0]) if frame is not None else None,
                "columns": int(frame.shape[1]) if frame is not None else None,
                "memory_mb": round(memory_usage_mb(frame), 2) if frame is not None else None,
            }
            for table_name, frame in history_frame_map.items()
        ]
    )
    display(history_table_overview)

    descriptive_stats = (
        analysis_df.describe(include="all")
        .transpose()
        .reset_index()
        .rename(columns={"index": "column"})
    )
    correlation_artifact = (
        corr_df.reset_index().rename(columns={"index": "column"})
        if not corr_df.empty
        else pd.DataFrame(columns=["column"])
    )

    artifact_frames = {
        "table_availability.csv": table_summary,
        "parameter_summary.csv": parameter_summary,
        "application_train_summary.csv": main_table_summary,
        "duplicate_key_summary.csv": duplicate_key_summary,
        "derived_notes.csv": derived_note_frame,
        "application_train_descriptive_stats.csv": descriptive_stats,
        "dtype_summary.csv": dtype_summary,
        "key_business_column_availability.csv": availability_df,
        "target_summary.csv": target_summary,
        "top_missingness.csv": top_missing,
        "numeric_distribution_summary.csv": numeric_summary,
        "numeric_by_target_median.csv": grouped_numeric_summary,
        "correlation_matrix.csv": correlation_artifact,
        "outlier_summary.csv": outlier_summary,
        "fairness_summary.csv": fairness_summary,
        "age_band_summary.csv": age_summary,
        "annuity_income_band_summary.csv": ratio_summary,
        "history_table_focus.csv": history_table_focus,
        "history_table_overview.csv": history_table_overview,
        "checkpoint.csv": checkpoint,
    }

    EDA_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    artifact_rows = []
    for filename, frame in artifact_frames.items():
        artifact_path = EDA_OUTPUT_ROOT / filename
        frame.to_csv(artifact_path, index=False)
        artifact_rows.append({"artifact": filename.replace(".csv", ""), "path": str(artifact_path)})

    eda_summary = {
        "phase": 1,
        "ready": True,
        "raw_data_dir": str(RAW_DATA_DIR),
        "eda_output_root": str(EDA_OUTPUT_ROOT),
        "figure_directory": str(FIGURE_DIR),
        "figure_prefix": FIGURE_PREFIX,
        "export_figures": bool(EXPORT_FIGURES),
        "application_train_rows": int(analysis_df.shape[0]),
        "application_train_columns": int(analysis_df.shape[1]),
        "observed_default_rate": round(float(analysis_df[settings.target_column].mean()), 6)
        if settings.target_column in analysis_df.columns
        else None,
        "available_history_tables": available_history,
        "missing_history_tables": missing_history,
        "artifacts": artifact_rows,
    }
    eda_summary_path = EDA_OUTPUT_ROOT / "eda_summary.json"
    eda_summary_path.write_text(json.dumps(eda_summary, indent=2, ensure_ascii=False), encoding="utf-8")
    artifact_rows.append({"artifact": "eda_summary", "path": str(eda_summary_path)})

    phase1_artifact_manifest = pd.DataFrame(artifact_rows)
    display(phase1_artifact_manifest)
    print("Phase 1 完成边界：这里结束于描述性分析，不在本 notebook 中构建客户级训练表。")


In [ ]:
# Phase 1 report upgrade add-ons
from credit_visable.features import compute_iv_summary, compute_woe_detail
from credit_visable.utils import (
    REPORT_COLOR_PALETTE,
    add_conclusion_annotation,
    build_report_summary_fields,
    to_builtin,
)

iv_summary = pd.DataFrame()
age_woe_detail = pd.DataFrame()
age_income_default_heatmap = pd.DataFrame()
income_family_default_heatmap = pd.DataFrame()

if not PHASE_1_READY:
    print("Phase 1 report upgrade skipped because application_train.csv is unavailable.")
else:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    EDA_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    phase1_upgrade_figure_paths = {}
    phase1_upgrade_artifacts = []

    iv_candidate_columns = existing_columns(
        analysis_df,
        [
            settings.target_column,
            "AGE_YEARS",
            "YEARS_EMPLOYED",
            "AMT_INCOME_TOTAL",
            "AMT_CREDIT",
            "AMT_ANNUITY",
            "INCOME_CREDIT_RATIO",
            "ANNUITY_INCOME_RATIO",
            "EXT_SOURCE_1",
            "EXT_SOURCE_2",
            "EXT_SOURCE_3",
            "NAME_FAMILY_STATUS",
        ],
    )
    if len(iv_candidate_columns) > 1:
        iv_summary = compute_iv_summary(
            analysis_df[iv_candidate_columns].copy(),
            target_column=settings.target_column,
            bins=10,
        )
        iv_path = EDA_OUTPUT_ROOT / "iv_summary.csv"
        iv_summary.to_csv(iv_path, index=False)
        phase1_upgrade_artifacts.append({"artifact": "iv_summary", "path": str(iv_path)})

        top_iv = iv_summary.head(20).sort_values("iv", ascending=True)
        fig, ax = plt.subplots(figsize=(9, max(5, 0.35 * len(top_iv))))
        ax.barh(top_iv["feature"], top_iv["iv"], color=REPORT_COLOR_PALETTE["bad"])
        ax.set_title("Top IV Features - Strongest Risk Separation Signals")
        ax.set_xlabel("Information Value")
        ax.set_ylabel("Feature")
        if not iv_summary.empty:
            add_conclusion_annotation(ax, f"Top feature: {iv_summary.iloc[0]['feature']}")
        fig.tight_layout()
        phase1_upgrade_figure_paths["top20_iv_features"] = FIGURE_DIR / f"{FIGURE_PREFIX}top20_iv_features.png"
        fig.savefig(phase1_upgrade_figure_paths["top20_iv_features"], dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)

    if {"AGE_YEARS", settings.target_column}.issubset(analysis_df.columns):
        age_woe_detail = compute_woe_detail(
            analysis_df[[settings.target_column, "AGE_YEARS"]].copy(),
            target_column=settings.target_column,
            feature_column="AGE_YEARS",
            bins=6,
        )
        age_woe_path = EDA_OUTPUT_ROOT / "age_woe_detail.csv"
        age_woe_detail.to_csv(age_woe_path, index=False)
        phase1_upgrade_artifacts.append({"artifact": "age_woe_detail", "path": str(age_woe_path)})

        plot_frame = age_woe_detail.loc[age_woe_detail["bin"] != "Missing"].copy()
        if not plot_frame.empty:
            highest_risk_bin = plot_frame.sort_values("event_rate", ascending=False).iloc[0]["bin"]
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(plot_frame["bin_order"], plot_frame["woe"], marker="o", linewidth=2, color=REPORT_COLOR_PALETTE["accent"])
            ax.axhline(0.0, linestyle="--", color=REPORT_COLOR_PALETTE["neutral"], linewidth=1)
            ax.set_title("Age WOE Trend - Risk Separation by Age Band")
            ax.set_xlabel("Age bin")
            ax.set_ylabel("WOE")
            ax.set_xticks(plot_frame["bin_order"])
            ax.set_xticklabels(plot_frame["bin"], rotation=45, ha="right")
            add_conclusion_annotation(ax, f"Highest bad-rate bin: {highest_risk_bin}")
            fig.tight_layout()
            phase1_upgrade_figure_paths["age_band_woe_trend"] = FIGURE_DIR / f"{FIGURE_PREFIX}age_band_woe_trend.png"
            fig.savefig(phase1_upgrade_figure_paths["age_band_woe_trend"], dpi=150, bbox_inches="tight")
            plt.show()
            plt.close(fig)

    if {"AGE_YEARS", "AMT_INCOME_TOTAL", settings.target_column}.issubset(analysis_df.columns):
        heatmap_frame = analysis_df[["AGE_YEARS", "AMT_INCOME_TOTAL", settings.target_column]].copy()
        heatmap_frame = heatmap_frame.dropna()
        heatmap_frame = heatmap_frame.loc[heatmap_frame["AGE_YEARS"].between(18, 100)]
        if not heatmap_frame.empty:
            heatmap_frame["age_band"] = pd.cut(
                heatmap_frame["AGE_YEARS"],
                bins=[18, 25, 35, 45, 55, 65, 100],
                labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
                include_lowest=True,
                right=False,
            )
            income_rank = heatmap_frame["AMT_INCOME_TOTAL"].rank(method="first")
            income_bins = min(5, int(income_rank.nunique()))
            if income_bins >= 2:
                heatmap_frame["income_band"] = pd.qcut(
                    income_rank,
                    q=income_bins,
                    labels=[f"Q{i}" for i in range(1, income_bins + 1)],
                    duplicates="drop",
                )
                age_income_default_heatmap = (
                    heatmap_frame.groupby(["age_band", "income_band"], observed=False)[settings.target_column]
                    .mean()
                    .unstack()
                )
                age_income_path = EDA_OUTPUT_ROOT / "age_income_default_heatmap.csv"
                age_income_default_heatmap.reset_index().to_csv(age_income_path, index=False)
                phase1_upgrade_artifacts.append({"artifact": "age_income_default_heatmap", "path": str(age_income_path)})
                fig, ax = plt.subplots(figsize=(8, 5))
                sns.heatmap(age_income_default_heatmap, annot=True, fmt=".2%", cmap="Reds", ax=ax)
                ax.set_title("Age x Income Default Heatmap")
                ax.set_xlabel("Income band")
                ax.set_ylabel("Age band")
                if age_income_default_heatmap.size > 0:
                    max_position = np.nanargmax(age_income_default_heatmap.to_numpy())
                    row_index, column_index = np.unravel_index(max_position, age_income_default_heatmap.shape)
                    add_conclusion_annotation(
                        ax,
                        f"Highest risk cell: {age_income_default_heatmap.index[row_index]} / {age_income_default_heatmap.columns[column_index]}",
                    )
                fig.tight_layout()
                phase1_upgrade_figure_paths["age_income_default_heatmap"] = FIGURE_DIR / f"{FIGURE_PREFIX}age_income_default_heatmap.png"
                fig.savefig(phase1_upgrade_figure_paths["age_income_default_heatmap"], dpi=150, bbox_inches="tight")
                plt.show()
                plt.close(fig)

    if {"AMT_INCOME_TOTAL", "NAME_FAMILY_STATUS", settings.target_column}.issubset(analysis_df.columns):
        family_frame = analysis_df[["AMT_INCOME_TOTAL", "NAME_FAMILY_STATUS", settings.target_column]].copy().dropna()
        if not family_frame.empty:
            top_families = family_frame["NAME_FAMILY_STATUS"].value_counts().head(5).index.tolist()
            family_frame = family_frame.loc[family_frame["NAME_FAMILY_STATUS"].isin(top_families)].copy()
            income_rank = family_frame["AMT_INCOME_TOTAL"].rank(method="first")
            income_bins = min(5, int(income_rank.nunique()))
            if income_bins >= 2:
                family_frame["income_band"] = pd.qcut(
                    income_rank,
                    q=income_bins,
                    labels=[f"Q{i}" for i in range(1, income_bins + 1)],
                    duplicates="drop",
                )
                income_family_default_heatmap = (
                    family_frame.groupby(["NAME_FAMILY_STATUS", "income_band"], observed=False)[settings.target_column]
                    .mean()
                    .unstack()
                )
                income_family_path = EDA_OUTPUT_ROOT / "income_family_default_heatmap.csv"
                income_family_default_heatmap.reset_index().to_csv(income_family_path, index=False)
                phase1_upgrade_artifacts.append({"artifact": "income_family_default_heatmap", "path": str(income_family_path)})
                fig, ax = plt.subplots(figsize=(8, 5))
                sns.heatmap(income_family_default_heatmap, annot=True, fmt=".2%", cmap="Reds", ax=ax)
                ax.set_title("Income x Family Status Default Heatmap")
                ax.set_xlabel("Income band")
                ax.set_ylabel("Family status")
                fig.tight_layout()
                phase1_upgrade_figure_paths["income_family_default_heatmap"] = FIGURE_DIR / f"{FIGURE_PREFIX}income_family_default_heatmap.png"
                fig.savefig(phase1_upgrade_figure_paths["income_family_default_heatmap"], dpi=150, bbox_inches="tight")
                plt.show()
                plt.close(fig)

    summary_path = EDA_OUTPUT_ROOT / "eda_summary.json"
    existing_summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
    top_iv_feature = iv_summary.iloc[0]["feature"] if not iv_summary.empty else "n/a"
    key_findings = [
        f"Top IV feature: {top_iv_feature}.",
        "Phase 1 now includes WOE trend diagnostics for age and two interaction heatmaps for business segmentation.",
        "Interaction views help move the phase from descriptive slices toward feature-engineering insight.",
    ]
    business_implications = [
        "Top IV features provide a defensible shortlist for Phase 2 feature governance and later explainability review.",
        "WOE trend checks help identify monotonic and non-monotonic risk patterns before scorecard-style interpretation.",
        "Interaction heatmaps highlight concentrated high-risk cells that matter more than single-variable averages.",
    ]
    existing_summary.update(
        build_report_summary_fields(
            headline="Phase 1 EDA now includes IV ranking, WOE diagnostics, and interaction risk views.",
            key_findings=key_findings,
            business_implications=business_implications,
            figure_paths={path.stem: path for path in sorted(FIGURE_DIR.glob(f"{FIGURE_PREFIX}*.png"))},
        )
    )
    existing_artifacts = existing_summary.get("artifacts", [])
    existing_paths = {artifact.get("path") for artifact in existing_artifacts}
    existing_artifacts.extend(
        artifact for artifact in phase1_upgrade_artifacts if artifact["path"] not in existing_paths
    )
    existing_summary["artifacts"] = existing_artifacts
    summary_path.write_text(json.dumps(to_builtin(existing_summary), indent=2, ensure_ascii=False), encoding="utf-8")

    display(iv_summary.head(20))
    if not age_woe_detail.empty:
        display(age_woe_detail)
